#### 데이터 분석 컨설팅: 기장군청
* CCTV 신규설치 지역선정을 위한 지도시각화 분석

In [2]:
# 데이터 전처리 라이브러리, 데이터 경로 설정
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

REF_PATH = r"D:\\02_사업관리\\자체분석\\"
OUTPUT_PATH = r"03_집계결과\\"
# grid50_pg = gpd.read_file(DATA_PATH + r"04_공간DB\\griddata\\bigdata_50m_busan.shp").to_crs(5179)
grid50_pg = gpd.read_file(REF_PATH + r"04_공간DB\\griddata\\busan_50cell_v15.shp").set_crs(5179)
grid50_pg['X'] = grid50_pg.geometry.centroid.x.round(3)
grid50_pg['Y'] = grid50_pg.geometry.centroid.y.round(3)
grid50_pt = gpd.GeoDataFrame(grid50_pg, geometry=gpd.points_from_xy(grid50_pg['X'], grid50_pg['Y']))
grid100_pg = gpd.read_file(REF_PATH + r"04_공간DB\\griddata\\ngii_100m_busan.shp").to_crs(5179)
emd26710 = gpd.read_file(r"2026-022_26710_CCTV.gpkg", layer="EMD26710")
grid100pg_26710 = gpd.overlay(grid100_pg, emd26710, how="intersection", keep_geom_type=True)
hjdong = pd.read_csv(REF_PATH + r"05_코드자료\\부산행정동코드.csv", dtype={'hcode': str}, encoding="utf-8")
time_dicts = dict((f'{h:02d}H', [f'h_t_{h:02d}', f'w_t_{h:02d}', f'v_t_{h:02d}']) for h in range(24))
keys = [key for key in time_dicts.keys()]
slots = ['심야2','오전1','오전2','오후1','오후2','심야1']
time_ranges = {f'{h:02d}H': slots[h//4] for h in range(24)}

In [3]:
time_ranges

{'00H': '심야2',
 '01H': '심야2',
 '02H': '심야2',
 '03H': '심야2',
 '04H': '오전1',
 '05H': '오전1',
 '06H': '오전1',
 '07H': '오전1',
 '08H': '오전2',
 '09H': '오전2',
 '10H': '오전2',
 '11H': '오전2',
 '12H': '오후1',
 '13H': '오후1',
 '14H': '오후1',
 '15H': '오후1',
 '16H': '오후2',
 '17H': '오후2',
 '18H': '오후2',
 '19H': '오후2',
 '20H': '심야1',
 '21H': '심야1',
 '22H': '심야1',
 '23H': '심야1'}

1. 2022년~2026년 기장군 생활인구 전처리
- 2026년은 7월까지 적용

In [6]:
# 100m격자기준 생활인구 전처리
for year in tqdm(range(2023, 2027)):
    print(f"{year}년 격자별 생활인구 집계처리 시작")
    df = pd.read_csv(REF_PATH + f"01_생활인구\\l0kte_kt_d_living_tme_p2_{year}.csv", sep="|", 
                                dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
    df['std_year'] = df['std_ymd'].str[:4] + "년"
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    # merged = pd.merge(df, grid50_pg, on="id", how="left")
    # merged.drop(columns=['geometry'], inplace=True)
    # merged = gpd.GeoDataFrame(merged, geometry=gpd.points_from_xy(merged.X, merged.Y)).set_crs(5179)
    # sjoined = gpd.sjoin(grid100_pg, merged, how="inner", predicate="intersects")
    merged = pd.merge(df, grid100_pg, left_on="id", right_on="gid", how="left")
    merged = gpd.GeoDataFrame(merged, geometry="geometry")
    for key, value in time_dicts.items():
        # sjoined[key] = sjoined.loc[:, value].sum(axis=1) / 3
        merged[key] = merged.loc[:, value].sum(axis=1) / 3
    agg_dict = {key: "sum" for key in time_dicts.keys()}
    keys = [key for key in time_dicts.keys()]
    groups = merged.dissolve(by=['std_year','hcode','hname','gid'],
                                            aggfunc=agg_dict).reset_index().sort_values(by=['std_year','hcode'],
                                                                                        ascending=True)
    melts = groups.melt(id_vars=['std_year','hname','gid','geometry'], value_vars=keys, var_name='times', value_name='pop')
    melts['시간대'] = melts['times'].map(time_ranges)
    melts = melts[['std_year','hname','gid','times','시간대','pop','geometry']]
    melts.rename(columns={'std_year': '기준연도', 'hname': '행정동명', 'gid': '100MGID', 
                          'times': '특정시간', '시간대': '시간분류', 'pop': '생활인구'}, inplace=True)
    melts.to_file(r"2026-022_26710_CCTV.gpkg", layer=f"100m_hjdong_pop_{year}")
    print(f"{year}년 격자별 생활인구 집계처리 완료")

  0%|          | 0/4 [00:00<?, ?it/s]

2023년 격자별 생활인구 집계처리 시작


 25%|██▌       | 1/4 [04:13<12:40, 253.54s/it]

2023년 격자별 생활인구 집계처리 완료
2024년 격자별 생활인구 집계처리 시작


 50%|█████     | 2/4 [08:06<08:02, 241.45s/it]

2024년 격자별 생활인구 집계처리 완료
2025년 격자별 생활인구 집계처리 시작


 75%|███████▌  | 3/4 [12:13<04:04, 244.08s/it]

2025년 격자별 생활인구 집계처리 완료
2026년 격자별 생활인구 집계처리 시작


100%|██████████| 4/4 [14:31<00:00, 217.99s/it]

2026년 격자별 생활인구 집계처리 완료


2. 2022년~2026년 유입지 생활인구 전처리

In [1]:
import pandas as pd
import calendar
import glob
pd.set_option("display.max_rows", None)
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

vals = []
months = [f"{y}{month:02d}" for y in range(2022, 2027) for month in range(1, 13)]
for ym in months:
    vals.append({'기준연월': ym, '월별일수': calendar.monthrange(int(ym[:4]), int(ym[-2:]))[1]})
monthly = pd.DataFrame(vals)
monthly['기준연도'] = monthly['기준연월'].str[:4] + "년"
yearly = monthly.groupby('기준연도')['월별일수'].sum().reset_index(name="연별일수")

DATA_PATH = r"D:\\02_사업관리\\자체분석\\01_생활인구\\"
CODE_PATH = r"D:\\02_사업관리\\자체분석\\05_코드자료\\"
OUTPUT_PATH = r"03_집계결과\\"
hjdong = pd.read_csv(CODE_PATH + r"행정경계코드\\KIKcd_H20260301_include_oldcode.csv",
                                 dtype={'행정동코드': str}, encoding="utf-8")

# 관내, 관외 행정코드 전처리
hjdong = hjdong[(~hjdong['시군구명'].isna()) & (~hjdong['읍면동명'].isna())]
hjdcode = hjdong[hjdong['행정동코드'].str.startswith('26710')].copy()
hjdcode.rename(columns={'행정동코드': 'hcode'}, inplace=True)
hjdcode = hjdcode[['hcode','시군구명','읍면동명']]
hjdcode.rename(columns={'시군구명': 'gname1', '읍면동명': 'hjdname1'}, inplace=True)
INhjdong = hjdong[(hjdong['행정동코드'].str.startswith('26')) & (hjdong['행정동코드'].str[:5] != '26710')]
INhjdong['inflow_cd'] = INhjdong['행정동코드'].str[:5] + "00000"
INhjdong = INhjdong[['inflow_cd','행정동코드','시군구명','읍면동명','생성일자','말소일자']]
INhjdong.rename(columns={'시군구명': 'gname2', '읍면동명': 'hjdname2'}, inplace=True)
OUThjdong = hjdong[~hjdong['행정동코드'].str.startswith('26')]
OUThjdong['inflow_cd'] = OUThjdong['행정동코드'].str[:2] + "00000000"
OUThjdong = OUThjdong[['inflow_cd','시도명','시군구명','읍면동명','생성일자','말소일자']]
OUThjdong.rename(columns={'시도명': 'sname2', '시군구명': 'gname2', '읍면동명': 'hjdname2'}, inplace=True)
slots = ['심야2','오전1','오전2','오후1','오후2','심야1']
time_ranges = {f'{h:02d}H': slots[h//4] for h in range(24)}

In [ ]:

# 부산관내 유입인구 집계
values = []
for year in tqdm(range(2022, 2027)):
    df = pd.read_csv(DATA_PATH + f"l0kte_kt_d_living_tme_inflow_{year}.csv", sep="|", 
                                 dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str, 'tme': str, 'time': str}, 
                                 encoding="utf-8")
    df.drop(columns=['dat_flow_id','part_batchdate'], inplace=True)
    if 'time' in df.columns:
        df.rename(columns={'time': 'tme'}, inplace=True)
    df = df[(df['hcode'].str.startswith('26')) & (df['inflow_cd'].str.startswith('26')) & (df['inflow_cd'].str[:5] != '26710')]
    df['기준연도'] = df['std_ymd'].str[:4] + "년"
    df['inflow_cd'] = df['inflow_cd'].str[:5] + "00000"
    df['시간'] = df['tme'] + "H"
    df['시간대'] = df['시간'].map(time_ranges)
    df = df.merge(hjdcode[['hcode','gname1','hjdname1']], on="hcode", how="left")
    df = df.merge(INhjdong[['inflow_cd','gname2','hjdname2']], on="inflow_cd", how="left")
    df['sum_pop'] = df.loc[:, 'h_pop': 'v_pop'].sum(axis=1)
    values.append(df.groupby(['기준연도','hcode','시간대','hjdname1','gname2']
                             )['sum_pop'].sum().round().reset_index(name='유입인구').sort_values(['기준연도','hcode'], ascending=True))
concated = pd.concat(values, ignore_index=True)
concated.columns = ['기준연도','행정동코드','행정동명','시간대','유입구군','유입인구']
concated.to_csv(OUTPUT_PATH + "2_연도별_행정동별_유입인구 현황.csv", sep="|", encoding="utf-8", index=False)


100%|██████████| 5/5 [08:00<00:00, 96.08s/it] 


In [8]:
# 부산관외 유입인구 집계
values = []
for year in tqdm(range(2022, 2027)):
    df = pd.read_csv(DATA_PATH + f"l0kte_kt_d_living_tme_inflow_{year}.csv", sep="|", 
                                 dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str, 'tme': str, 'time': str}, 
                                 encoding="utf-8")
    df.drop(columns=['dat_flow_id','part_batchdate'], inplace=True)
    if 'time' in df.columns:
        df.rename(columns={'time': 'tme'}, inplace=True)
    df = df[~df['inflow_cd'].str.startswith('26')].copy()
    df['기준연도'] = df['std_ymd'].str[:4] + "년"
    df['inflow_cd'] = df['inflow_cd'].str[:2] + "00000000"
    df['시간'] = df['tme'] + "H"
    df['시간대'] = df['시간'].map(time_ranges)
    # df = df.merge(hjdcode[['hcode','hjdname1']], on="hcode", how="left")
    # df = df.merge(OUThjdong[['inflow_cd','sname2']], on="inflow_cd", how="left")
    df['sum_pop'] = df.loc[:, 'h_pop': 'v_pop'].sum(axis=1)
    values.append(df.groupby(['기준연도','hcode','시간대','inflow_cd'])['sum_pop'].sum().round().reset_index(name='유입인구').sort_values(['기준연도','hcode'], ascending=True))
concated = pd.concat(values, ignore_index=True)
concated = concated.merge(hjdcode[['hcode','hjdname1']], on="hcode", how="left")
concated = concated.merge(OUThjdong[['inflow_cd','sname2']], on="inflow_cd", how="left")
concated.rename(columns={'hjdname1': '행정동명', 'sname2': '유입시도'}, inplace=True)
concated.drop_duplicates(keep='first', inplace=True)
concated['유입시도'] = concated['유입시도'].fillna('기타')
concated = concated[['기준연도','행정동명','시간대','유입시도','유입인구']]
concated.to_csv(OUTPUT_PATH + "2_연도별_행정동별_관외유입인구 현황.csv", sep="|", encoding="utf-8", index=False)


100%|██████████| 5/5 [00:28<00:00,  5.60s/it]


3. CCTV 위치정보 확인

In [6]:
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_rows', None)

DATA_PATH = r"02_자료수집\\"
cctv = pd.read_excel(DATA_PATH + r"CCTV 위치정보(기장군)_중복제거_혜안변환.xlsx", engine="openpyxl")
cctv.loc[cctv['촬영방면정보'].str.contains('360도 전방향'), '촬영범위'] = 360
cctv.loc[cctv['촬영방면정보'].str.contains('60도 단방향'), '촬영범위'] = 60
cctv.loc[cctv['촬영방면정보'].str.contains('360도 전방향'), '감시거리'] = 15
cctv.loc[cctv['촬영방면정보'].str.contains('60도 단방향'), '감시거리'] = 30
cctv.loc[cctv['촬영방면정보'].str.contains('360도 전방향'), '비고'] = '무지향성'
cctv.loc[cctv['촬영방면정보'].str.contains('60도 단방향'), '비고'] = '특정방향'
cctv = gpd.GeoDataFrame(cctv, geometry=gpd.points_from_xy(cctv['경도'], cctv['위도']), crs=4326).to_crs(5179)
cctv.to_file(r"2026-022_26710_CCTV.gpkg", layer="CCTVINFO_26170", driver="GPKG")

In [1]:
import geopandas as gpd
g = gpd.read_file("2026-022_26710_CCTV.gpkg", layer="100m_hjdong_pop_2025", ignore_geometry=True)
print(g.columns.tolist()); print(len(g), g["100MGID"].nunique())
print(g.groupby(["100MGID", "시간분류"]).size().describe()); print(g.head())

['기준연도', '행정동명', '100MGID', '특정시간', '시간분류', '생활인구']
524640 21601
count    129606.000000
mean          4.047961
std           0.435367
min           4.000000
25%           4.000000
50%           4.000000
75%           4.000000
max           8.000000
dtype: float64
    기준연도 행정동명   100MGID 특정시간 시간분류        생활인구
0  2025년  기장읍  마라510915  00H  심야2  424.900000
1  2025년  기장읍  마라510916  00H  심야2    1.833333
2  2025년  기장읍  마라510917  00H  심야2    2.586667
3  2025년  기장읍  마라510918  00H  심야2    0.426667
4  2025년  기장읍  마라511914  00H  심야2  618.346667
